### INSTALAÇÕES - SE NECESSÁRIO -




In [1]:
# Instalar pacotes necessários (se ainda não instalados)
!pip install nltk scikit-learn

In [2]:
!pip install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.0 MB/s eta 0:00:00


### IMPORTAÇÕES

In [3]:
import glob
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import unidecode
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

In [4]:
# prompt: conecte ao drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# TRATAMENTO DOS DADOS

> Adicionar blockquote



In [5]:
# Configurar NLTK (stopwords)

# Baixar recursos do NLTK
nltk.download('stopwords')
nltk.download('rslp')

# Configurações
stopwords_pt = set(stopwords.words('portuguese'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


Opção 2 de pré processamento

In [9]:
# Lista de stopwords padrão do português
stopwords_pt = set(stopwords.words('portuguese'))

# Lista de palavras-chave (cidades e candidatos) a serem excluídas
palavras_excluir = {
    'aracaju', 'emilia', 'correa', 'luiz', 'roberto',
    'fortaleza', 'andre', 'fernandes', 'evandro', 'leitao',
    'joao', 'pessoa', 'cicero', 'lucena', 'luciano', 'cartaxo',
    'maceio', 'jhc', 'rafael', 'brito',
    'natal', 'natalia', 'bonavides', 'paulinho', 'freire',
    'recife', 'gilson', 'machado', 'neto', 'campos',
    'salvador', 'bruno', 'reis', 'geraldo', 'jr', 'kleber', 'rosa',
    'sao', 'luis', 'duarte', 'eduardo', 'braide',
    'teresina', 'fabio', 'novo', 'silvio', 'mendes', 'marcelo', 'queiroga', 'propaganda', 'eleitoral'
}

def preprocessar_texto(texto):
    if pd.isna(texto) or texto.strip() == '':
        return ''

    texto = texto.lower()

    # Remover URLs
    texto = re.sub(r"http\S+|www\S+", '', texto)

    # Remover hashtags e menções
    texto = re.sub(r'#\S+', '', texto)
    texto = re.sub(r'@\S+', '', texto)

    # Remover caracteres não alfabéticos
    texto = re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', texto)

    # Remover acentuação
    texto = unidecode.unidecode(texto)

    # Tokenização
    palavras = texto.split()

    # Remover stopwords e palavras a excluir
    palavras_filtradas = [p for p in palavras if p not in stopwords_pt and p not in palavras_excluir]

    return ' '.join(palavras_filtradas)


## Vencedor

### TRATAMENTO

In [15]:
# Carregar todos os JSONs da pasta
caminho_json = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/situacao_final/vencedor/*.json"
arquivos = glob.glob(caminho_json)

# Lista para armazenar os textos
df_vencedor_list = []

for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)
        df = pd.DataFrame(posts)
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)

        # Manter SOMENTE a coluna processada
        df = df[['texto_processado']]

        df_vencedor_list.append(df)


In [16]:
# Criar diretório para salvar os CSVs, se não existir
os.makedirs("dataset_resultado", exist_ok=True)

# Concatenar todos os DataFrames em um único DataFrame para vencedores
df_vencedor_final = pd.concat(df_vencedor_list, ignore_index=True)

# Salvar o DataFrame final como CSV
nome_arquivo_csv = "dataset_resultado/vencedor_processado.csv"
df_vencedor_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8')

print(f"Dataset dos Vencedores exportado para: {nome_arquivo_csv}")

Dataset dos Vencedores exportado para: dataset_resultado/vencedor_processado.csv


In [21]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf_vencedor = vectorizer.fit_transform(df_vencedor_final['texto_processado'])

# Obter termos filtrados
termos_filtrados_vencedor = vectorizer.get_feature_names_out()

In [23]:
# Converter a matriz esparsa TF-IDF para um DataFrame denso
df_tfidf_vencedor = pd.DataFrame(X_tfidf_vencedor.toarray(), columns=termos_filtrados_vencedor)

# Salvar o DataFrame TF-IDF como CSV
nome_arquivo_tfidf_csv = "dataset_resultado/dataset_TF-IDF_vencedor.csv"
df_tfidf_vencedor.to_csv(nome_arquivo_tfidf_csv, index=False, encoding='utf-8')

print(f"Dataset TF-IDF dos vencedores exportado para: {nome_arquivo_tfidf_csv}")

Dataset TF-IDF dos vencedores exportado para: dataset_resultado/dataset_TF-IDF_vencedor.csv


### WORD CLOUDS

In [24]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf_vencedor.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados_vencedor, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Vencedor", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/vencedor.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [25]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_vencedor.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")


## Perdedor

### TRATAMENTO

In [26]:
# Carregar todos os JSONs da pasta
caminho_json = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/situacao_final/perdedor/*.json"
arquivos = glob.glob(caminho_json)

# Lista para armazenar os textos
df_perdedor_list = []

for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)
        df = pd.DataFrame(posts)
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)

        # Manter SOMENTE a coluna processada
        df = df[['texto_processado']]

        df_perdedor_list.append(df)

In [27]:
# Criar diretório para salvar os CSVs, se não existir
os.makedirs("dataset_resultado", exist_ok=True)

# Concatenar todos os DataFrames em um único DataFrame para vencedores
df_perdedor_final = pd.concat(df_perdedor_list, ignore_index=True)

# Salvar o DataFrame final como CSV
nome_arquivo_csv = "dataset_resultado/perdedor_processado.csv"
df_perdedor_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8')

print(f"Dataset dos Perdedores exportado para: {nome_arquivo_csv}")

Dataset dos Perdedores exportado para: dataset_resultado/perdedor_processado.csv


In [28]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf_perdedor = vectorizer.fit_transform(df_perdedor_final['texto_processado'])

# Obter termos filtrados
termos_filtrados_perdedor = vectorizer.get_feature_names_out()


In [29]:
# Converter a matriz esparsa TF-IDF para um DataFrame denso
df_tfidf_perdedor = pd.DataFrame(X_tfidf_perdedor.toarray(), columns=termos_filtrados_perdedor)

# Salvar o DataFrame TF-IDF como CSV
nome_arquivo_tfidf_csv = "dataset_resultado/dataset_TF-IDF_perdedor.csv"
df_tfidf_perdedor.to_csv(nome_arquivo_tfidf_csv, index=False, encoding='utf-8')

print(f"Dataset TF-IDF dos Perdedores exportado para: {nome_arquivo_tfidf_csv}")

Dataset TF-IDF dos Perdedores exportado para: dataset_resultado/dataset_TF-IDF_perdedor.csv


### WORDCLOUDS

In [30]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf_perdedor.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados_perdedor, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Perdedor", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/perdedor.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [31]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_perdedor.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")
